<a href="https://colab.research.google.com/github/riyamehta12/speech_impairement_recognition/blob/main/barkai.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!pip install git+https://github.com/suno-ai/bark.git

  Cloning https://github.com/suno-ai/bark.git to /tmp/pip-req-build-udkwtb1f
  Running command git clone --filter=blob:none --quiet https://github.com/suno-ai/bark.git /tmp/pip-req-build-udkwtb1f
  Resolved https://github.com/suno-ai/bark.git to commit f4f32d4cd480dfec1c245d258174bc9bde3c2148
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Using cached boto3-1.37.33-py3-none-any.whl.metadata (6.7 kB)
  Using cached encodec-0.1.1.tar.gz (3.7 MB)
  Preparing metadata (setup.py) ... done
  Using cached funcy-2.0-py2.py3-none-any.whl.metadata (5.9 kB)
  Using cached botocore-1.37.33-py3-none-any.whl.metadata (5.7 kB)
  Using cached jmespath-1.0.1-py3-none-any.whl.metadata (7.6 kB)
  Using cached s3transfer-0.11.4-py3-none-any.whl.metadata (1.7 kB)
  Using cached nvidia_cuda_nvrtc_cu12-12.4.127-py3-none-manylinux2014_x86_64.whl.metadata (1.5 kB)
  Using cached nvidia_cuda_runtime_cu12-12.4.127-py3-none-

In [3]:
!pip install git+https://github.com/huggingface/transformers.git

  Cloning https://github.com/huggingface/transformers.git to /tmp/pip-req-build-5lgawsfw
  Running command git clone --filter=blob:none --quiet https://github.com/huggingface/transformers.git /tmp/pip-req-build-5lgawsfw
  Resolved https://github.com/huggingface/transformers.git to commit 78cea3e22c71d820d0bb6fae46a0ee793084baca
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for transformers: filename=transformers-4.52.0.dev0-py3-none-any.whl size=11132601 sha256=ecca842c0e1eb7a1cb9cda10448e7620ef89a37477d70ed937fcf7935157be56
  Stored in directory: /tmp/pip-ephem-wheel-cache-j16s74rm/wheels/32/4b/78/f195c684dd3a9ed21f3b39fe8f85b48df7918581b6437be143
Successfully built transformers
  Attempting uninstall: transformers
    Found existing installation: transformers 4.50.3
    Uninstalling transformers-4.50.3:
      Successfully uninstalled transformers-4.50.3


In [8]:
from transformers import AutoProcessor, BarkModel
import scipy
import numpy as np
import os

# Load Bark model & processor
processor = AutoProcessor.from_pretrained("suno/bark")
model = BarkModel.from_pretrained("suno/bark")
model.eval()

# Ensure output directory
os.makedirs("generated_samples", exist_ok=True)

# Bark voice preset
voice_preset = "v2/en_speaker_6"
sample_rate = model.generation_config.sample_rate

# Healthy Prompts (fluent, well-structured speech)
healthy_prompts = [
    "Yesterday I visited the new art exhibit downtown and spent over an hour admiring the surrealist paintings and installations. The lighting and space were beautifully curated.",
    "I recently finished a book on behavioral psychology that explored how our habits shape our identity. The author used real-life stories to explain complex concepts clearly.",
    "Last weekend, I went hiking in the mountains with my friends. We followed a five-mile trail that offered breathtaking views of the valley and plenty of wildlife along the way.",
    "I spent the afternoon baking banana bread from scratch. It reminded me of the recipe my grandmother used, and the smell in the kitchen brought back childhood memories.",
    "Earlier today, I had a video call with my old college roommate. We talked about her new job in Paris, reminisced about our time in the dorms, and made plans to visit this summer."
]

# Impaired Prompts (simulating cognitive hesitation, word loss)
impaired_prompts = [
    "So, um, yesterday I... I think I was, uh, going to the place... the one with the big, uh... you know, the thing, uh, where we used to get coffee... or maybe it was groceries? Anyway, I... um, I forgot.",
    "I was talking to my... uh... my niece, or maybe it was my neighbor? Um... she said something about... oh, what was it... um, birthdays? No, no—uh, graduation. Yeah. I think.",
    "There was this movie I saw, um, the other day... it had that actor, uh, what’s his name... um... you know, he was in the... the thing with the spaceship. Uh... never mind, I lost it.",
    "Um... so I went out, I think it was Tuesday? Or Wednesday? Uh, I took the bus or the... train? Hmm... anyway, I ended up somewhere, but I don’t... I don’t really remember why I was there.",
    "Oh! I was gonna say something about my... uh... my brother? No, not brother—um, cousin! Yeah, he was in town, or maybe he called. Uh... either way, we talked about... um... family stuff, I think."
]

# Generate and save audio
def generate_and_save(prompt, index, label):
    inputs = processor(prompt, voice_preset=voice_preset, return_tensors="pt")

    # Add attention mask and pad token id to avoid warnings
    input_ids = inputs["input_ids"]
    attention_mask = inputs["attention_mask"]

    audio_array = model.generate(
        input_ids=input_ids,
        attention_mask=attention_mask,
        pad_token_id=10000  # same as eos_token_id for Bark
    ).cpu().numpy().squeeze()

    file_path = f"generated_samples/{label}_sample_{index+1}.wav"
    scipy.io.wavfile.write(file_path, rate=sample_rate, data=audio_array)
    print(f"Saved: {file_path}")

# Generate healthy samples
for i, text in enumerate(healthy_prompts):
    generate_and_save(text, i, "healthy")

# Generate impaired samples
for i, text in enumerate(impaired_prompts):
    generate_and_save(text, i, "impaired")


KeyboardInterrupt: 